In [1]:
import numpy as np
import feos
import si_units as si

import thermoift.FeosPlugin as TC
import thermoift.semi_emperical_correlations as SEC

COMPONENT_NAMES = ["carbon dioxide", "hydrogen", "argon"]
FEED_Z = np.array([0.95, 0.02, 0.03], dtype=float)
FEED_Z = FEED_Z / FEED_Z.sum()

T_K = 250.0

# Example placeholders: replace with your actual parachor values
PARACHOR_NUMBERS = np.array([
    77,   # CO2
    25,   # H2
    50,   # Ar
], dtype=float)

# optional kij matrix for parachor mixing
kij_parachor = np.zeros((3, 3), dtype=float)

params = TC.PARAMETERS(COMPONENT_NAMES)
eos = feos.HelmholtzEnergyFunctional.pcsaft(params)
feed_si = FEED_Z * si.MOL

T_vals = np.linspace(220.0, 300.0, 40)
T_bub, P_bub = TC.compute_bubble_curve(eos, T_vals, feed_si, verbose=True)
T_dew, P_dew = TC.compute_dew_curve(eos, T_vals, feed_si, verbose=True)

P_bub_T = np.interp(T_K, T_bub, P_bub)
P_dew_T = np.interp(T_K, T_dew, P_dew)
P_bar = 0.5 * (P_dew_T + P_bub_T)

eq, x, y, _, _ = TC.tp_flash(
    feos=feos,
    eos=eos,
    T=T_K * si.KELVIN,
    P=P_bar * si.BAR,
    feed=feed_si,
    molar_masses=TC.molar_masses(params),
)

rho_l = TC.molar_density_mol_m3(eq.liquid.density) * 1e-6
rho_v = TC.molar_density_mol_m3(eq.vapor.density) * 1e-6

model = SEC.semi_emperical_correlations()

gamma_parachor = model.parachor_mixture_IFT(
    x=x,
    y=y,
    rho_l=rho_l,
    rho_v=rho_v,
    parachor_numbers=PARACHOR_NUMBERS,
    kij=kij_parachor,
    n_exp=3.87,
)

print("x =", x)
print("y =", y)
print(f"Parachor gamma = {gamma_parachor:.6f} mN/m")

Bubble calculation failed at T = 297.94871794871796 K: Iteration resulted in trivial solution.
x = [0.96850724 0.00853878 0.02295398]
y = [0.54796122 0.26897576 0.18306302]
Parachor gamma = 7.310886 mN/m


In [2]:
import numpy as np
import feos
import si_units as si

import thermoift.FeosPlugin as TC
import thermoift.semi_emperical_correlations as SEC

COMPONENT_NAMES = ["carbon dioxide", "hydrogen", "argon"]
FEED_Z = np.array([0.95, 0.02, 0.03], dtype=float)
FEED_Z = FEED_Z / FEED_Z.sum()

T_K = 250.0

# -----------------------------------------
# Build EOS and choose a valid two-phase P
# -----------------------------------------
params = TC.PARAMETERS(COMPONENT_NAMES)
eos = feos.HelmholtzEnergyFunctional.pcsaft(params)
feed_si = FEED_Z * si.MOL

T_vals = np.linspace(220.0, 300.0, 40)
T_bub, P_bub = TC.compute_bubble_curve(eos, T_vals, feed_si, verbose=True)
T_dew, P_dew = TC.compute_dew_curve(eos, T_vals, feed_si, verbose=True)

P_bub_T = np.interp(T_K, T_bub, P_bub)
P_dew_T = np.interp(T_K, T_dew, P_dew)
P_bar = 0.5 * (P_dew_T + P_bub_T)

# -----------------------------------------
# TP flash
# -----------------------------------------
eq, x, y, _, _ = TC.tp_flash(
    feos=feos,
    eos=eos,
    T=T_K * si.KELVIN,
    P=P_bar * si.BAR,
    feed=feed_si,
    molar_masses=TC.molar_masses(params),
)

# molar densities in mol/cm3, exactly as expected by WSD/parachor
rho_l = TC.molar_density_mol_m3(eq.liquid.density) * 1e-6
rho_v = TC.molar_density_mol_m3(eq.vapor.density) * 1e-6

# -----------------------------------------
# WSD mixture IFT
# -----------------------------------------
model = SEC.semi_emperical_correlations()

# must be called first for WSD
gamma0, rhoL0, rhoV0, Tc0 = model.batch_pure_component_cDFT(COMPONENT_NAMES, T_K)

gamma_wsd, mixcorr = model.wsd_mixture_IFT(
    T_K=T_K,
    x=x,
    y=y,
    rho_l=rho_l,
    rho_v=rho_v,
    phi=1.0,          # scalar or NxN matrix
    correction=True,
)

print("x =", x)
print("y =", y)
print(f"rho_l = {rho_l:.8e} mol/cm3")
print(f"rho_v = {rho_v:.8e} mol/cm3")
print("pure gamma0 =", gamma0)
print("pure Tc     =", Tc0)
print(f"WSD gamma   = {gamma_wsd:.6f} mN/m")
print(f"mixcorr     = {mixcorr:.6f}")

Bubble calculation failed at T = 297.94871794871796 K: Iteration resulted in trivial solution.
x = [0.96850724 0.00853878 0.02295398]
y = [0.54796122 0.26897576 0.18306302]
rho_l = 2.36611160e-02 mol/cm3
rho_v = 2.14722339e-03 mol/cm3
pure gamma0 = [8.86799729        nan        nan]
pure Tc     = [309.14737606  65.50076682 150.29471988]
WSD gamma   = 7.901012 mN/m
mixcorr     = 0.968507
